In [4]:
!pip install -q langgraph langchain langchain-community chromadb google-generativeai
!pip install langchain chromadb pypdf langchain-google-genai langchain-community langchain-chroma groq -q

In [5]:
import zipfile
import os

try:
    zip_path = "/content/chroma_db_backup (1).zip"

    if not os.path.exists(zip_path):
        raise FileNotFoundError("Zip file not found — please upload chroma_db_backup (1).zip via the 📁 Files panel first")

    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall("/content/")

    if os.path.exists("/content/chroma_db"):
        print("✅ chroma_db restored successfully")
    else:
        raise FileNotFoundError("chroma_db folder not found after unzip")

except FileNotFoundError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Failed to unzip: {e}")

✅ chroma_db restored successfully


In [6]:
import os

try:
    if not os.path.exists("/content/chroma_db"):
        raise FileNotFoundError("chroma_db folder missing — run Cell 1 first")

    files = os.listdir("/content/chroma_db")
    if not files:
        raise ValueError("chroma_db folder is empty")

    print(f"✅ chroma_db folder found with {len(files)} file(s):")
    for f in files:
        print(f"   — {f}")

except FileNotFoundError as e:
    print(f"❌ {e}")
except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Unexpected error: {e}")

✅ chroma_db folder found with 2 file(s):
   — chroma.sqlite3
   — d30cd113-ae9c-4e6b-a635-c3a7018139b5


In [7]:
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from groq import Groq
import os

try:
    os.environ["GOOGLE_API_KEY"] = userdata.get("GEMINI_KEY")
    GROQ_KEY = userdata.get("GROQ_KEY")

    if not os.environ["GOOGLE_API_KEY"]:
        raise ValueError("GEMINI_KEY is empty — check Colab Secrets")
    if not GROQ_KEY:
        raise ValueError("GROQ_KEY is empty — check Colab Secrets")

    print("✅ API keys loaded")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Failed to load keys: {e}")

try:
    embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    print("✅ Embeddings model loaded")

except Exception as e:
    print(f"❌ Failed to load embeddings: {e}")

try:
    vectorstore = Chroma(
        persist_directory="/content/chroma_db",
        embedding_function=embeddings
    )

    count = vectorstore._collection.count()
    if count == 0:
        raise ValueError("ChromaDB is empty — re-upload and unzip chroma_db_backup.zip")

    print(f"✅ Vector store loaded — {count} chunks available")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Failed to load ChromaDB: {e}")

try:
    client = Groq(api_key=GROQ_KEY)
    print("✅ Groq client initialized")

except Exception as e:
    print(f"❌ Failed to initialize Groq: {e}")

✅ API keys loaded
✅ Embeddings model loaded
✅ Vector store loaded — 297 chunks available
✅ Groq client initialized


In [8]:
from typing import TypedDict, List

try:
    class RAGState(TypedDict):
        question: str      # input question from user
        context: str       # retrieved chunks joined together
        answer: str        # final answer from Groq
        sources: List[str] # individual chunk texts

    print("✅ RAGState defined successfully")

except Exception as e:
    print(f"❌ Failed to define state: {e}")

✅ RAGState defined successfully


In [9]:
try:
    def retrieve(state: RAGState):
        question = state["question"]

        if not question or not question.strip():
            raise ValueError("Question is empty")

        retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
        relevant_chunks = retriever.invoke(question)

        if not relevant_chunks:
            raise ValueError("No relevant chunks found for this question")

        context = "\n\n".join([doc.page_content for doc in relevant_chunks])
        sources = [doc.page_content for doc in relevant_chunks]

        print(f"✅ Retrieved {len(relevant_chunks)} chunks")
        return {"context": context, "sources": sources}

    print("✅ Retrieve node defined")

except Exception as e:
    print(f"❌ Failed to define retrieve node: {e}")

✅ Retrieve node defined


In [10]:
try:
    def generate(state: RAGState):
        question = state["question"]
        context = state["context"]

        if not context:
            raise ValueError("Context is empty — retrieve node may have failed")

        prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information."

Context:
{context}

Question: {question}
Answer:"""

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}]
        )

        answer = response.choices[0].message.content
        print(f"✅ Answer generated")
        return {"answer": answer}

    print("✅ Generate node defined")

except Exception as e:
    print(f"❌ Failed to define generate node: {e}")

✅ Generate node defined


In [11]:
from langgraph.graph import StateGraph, END

try:
    # Create the graph
    graph = StateGraph(RAGState)

    # Add nodes
    graph.add_node("retrieve", retrieve)
    graph.add_node("generate", generate)

    # Set the entry point
    graph.set_entry_point("retrieve")

    # Connect nodes
    graph.add_edge("retrieve", "generate")
    graph.add_edge("generate", END)

    # Compile
    rag_pipeline = graph.compile()

    print("✅ LangGraph pipeline compiled successfully")
    print("   Flow: retrieve → generate → END")

except Exception as e:
    print(f"❌ Failed to build graph: {e}")

✅ LangGraph pipeline compiled successfully
   Flow: retrieve → generate → END


In [12]:
try:
    test_question = "What is deep learning?"

    print(f"❓ Question: {test_question}\n")
    result = rag_pipeline.invoke({"question": test_question})

    print("🤖 Answer:")
    print(result["answer"])

    print("\n--- Retrieved Chunks ---")
    for i, chunk in enumerate(result["sources"], 1):
        print(f"\nChunk {i}:\n{chunk[:300]}...")

except KeyError as e:
    print(f"❌ Missing key in result: {e}")
except Exception as e:
    print(f"❌ Pipeline failed: {e}")

❓ Question: What is deep learning?

✅ Retrieved 5 chunks
✅ Answer generated
🤖 Answer:
Deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is used to transform input data into a progressively more abstract and composite representation. It involves utilizing multilayered neural networks to perform tasks such as classification, regression, and representation learning, with the "deep" referring to the use of multiple layers (ranging from three to several hundred or thousands) in the network.

--- Retrieved Chunks ---

Chunk 1:
The word "deep" in "deep learning" refers to the number of layers through which the data is transformed. More precisely, deep learning systems have a substantial credit assignment path (CAP) depth. The CAP is the chain of transformations from input to output. CAPs describe potentially causal connect...

Chunk 2:
Overview
Most modern deep learning models are based on multi-layered neural networks such as convolutional neural n

RAGAS

In [13]:
import subprocess

try:
    # Uninstall conflicting packages
    subprocess.run(["pip", "uninstall", "ragas", "langchain-core",
                   "langchain", "langchain-community",
                   "langchain-google-genai", "-y"],
                   capture_output=True, text=True)

    # Install exact compatible versions
    subprocess.run(["pip", "install",
                   "langchain-core==0.2.38",
                   "langchain==0.2.16",
                   "langchain-community==0.2.16",
                   "langchain-google-genai==1.0.10",
                   "langchain-chroma==0.1.4",
                   "langchain-text-splitters==0.2.4",
                   "langgraph==0.2.28",
                   "ragas==0.1.21",
                   "groq",
                   "datasets",
                   "-q"],
                   capture_output=True, text=True)

    print("✅ All packages reinstalled")
    print("⚠️ Restart now: Runtime → Restart Session → Yes")
except Exception as e:
    print(f"❌ Failed: {e}")

✅ All packages reinstalled
⚠️ Restart now: Runtime → Restart Session → Yes


In [14]:
!pip install -q ragas datasets

In [15]:
try:
    test_data = [
        {"question": "What is machine learning?", "ground_truth": "Machine learning is a field of computer science that gives computers the ability to learn without being explicitly programmed."},
        {"question": "What is deep learning?", "ground_truth": "Deep learning is a class of machine learning algorithms that uses multiple layers of neural networks to transform input data into progressively more abstract representations."},
        {"question": "What is a neural network?", "ground_truth": "A neural network is a group of interconnected units called neurons that send signals to one another to perform complex tasks."},
        {"question": "What is natural language processing?", "ground_truth": "Natural language processing is a field of AI that focuses on enabling computers to understand, interpret and generate human language."},
        {"question": "What is a transformer model?", "ground_truth": "A transformer model is a deep learning architecture that uses self-attention mechanisms to process sequential data like text."},
        {"question": "What is supervised learning?", "ground_truth": "Supervised learning is a type of machine learning where the model is trained on labeled data with input-output pairs."},
        {"question": "What is unsupervised learning?", "ground_truth": "Unsupervised learning is a type of machine learning where the model finds patterns in data without labeled examples."},
        {"question": "What is reinforcement learning?", "ground_truth": "Reinforcement learning is a type of machine learning where an agent learns by interacting with an environment and receiving rewards or penalties."},
        {"question": "What is backpropagation?", "ground_truth": "Backpropagation is an algorithm used to train neural networks by calculating gradients and updating weights to minimize error."},
        {"question": "What is an activation function?", "ground_truth": "An activation function determines the output of a neural network node given an input or set of inputs."},
        {"question": "What is overfitting in machine learning?", "ground_truth": "Overfitting is when a model learns the training data too well including noise, causing poor performance on new unseen data."},
        {"question": "What is a convolutional neural network?", "ground_truth": "A convolutional neural network is a type of deep learning model designed to process grid-like data such as images using convolutional layers."},
        {"question": "What is transfer learning?", "ground_truth": "Transfer learning is a technique where a model trained on one task is reused as the starting point for a model on a different task."},
        {"question": "What is the attention mechanism in transformers?", "ground_truth": "The attention mechanism allows transformers to weigh the importance of different words in a sequence when making predictions."},
        {"question": "What is a recurrent neural network?", "ground_truth": "A recurrent neural network is a type of neural network designed to work with sequential data by maintaining a hidden state across time steps."},
        {"question": "What is gradient descent?", "ground_truth": "Gradient descent is an optimization algorithm that minimizes a function by iteratively moving in the direction of steepest descent."},
        {"question": "What is word embedding?", "ground_truth": "Word embedding is a technique that represents words as dense vectors of real numbers capturing semantic relationships between words."},
        {"question": "What is a language model?", "ground_truth": "A language model is a type of machine learning model that learns to predict the probability of a sequence of words."},
        {"question": "What is tokenization in NLP?", "ground_truth": "Tokenization is the process of splitting text into smaller units called tokens such as words or subwords for processing by NLP models."},
        {"question": "What is the difference between AI and machine learning?", "ground_truth": "AI is the broader concept of machines performing tasks intelligently while machine learning is a specific subset of AI that learns from data."}
    ]
    print(f"✅ {len(test_data)} test questions defined")
except Exception as e:
    print(f"❌ Failed to define test data: {e}")

✅ 20 test questions defined


In [16]:
try:
    questions = []
    answers = []
    contexts = []
    ground_truths = []

    for i, item in enumerate(test_data):
        print(f"⏳ Running question {i+1}/20: {item['question'][:50]}...")

        try:
            result = rag_pipeline.invoke({"question": item["question"]})

            questions.append(item["question"])
            answers.append(result["answer"])
            contexts.append(result["sources"])
            ground_truths.append(item["ground_truth"])

            print(f"✅ Question {i+1} done")

        except Exception as e:
            print(f"❌ Question {i+1} failed: {e}")
            questions.append(item["question"])
            answers.append("Failed to generate answer")
            contexts.append([])
            ground_truths.append(item["ground_truth"])

    print(f"\n✅ All 20 questions processed")

except Exception as e:
    print(f"❌ Unexpected error: {e}")

⏳ Running question 1/20: What is machine learning?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 1 done
⏳ Running question 2/20: What is deep learning?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 2 done
⏳ Running question 3/20: What is a neural network?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 3 done
⏳ Running question 4/20: What is natural language processing?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 4 done
⏳ Running question 5/20: What is a transformer model?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 5 done
⏳ Running question 6/20: What is supervised learning?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 6 done
⏳ Running question 7/20: What is unsupervised learning?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 7 done
⏳ Running question 8/20: What is reinforcement learning?...
✅ Retrieved 5 chunks
✅ Answer generated
✅ Question 8 done
⏳ Running question 9/20: What is backpropagation?...
✅ Retrieved 5 chunks
✅ Ans

In [17]:
from datasets import Dataset

try:
    if not questions:
        raise ValueError("No questions found — run previous cell first")

    ragas_dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths
    })

    print(f"✅ RAGAS dataset created — {len(ragas_dataset)} rows")
    print(f"\nSample row:")
    print(f"  Question:  {ragas_dataset[0]['question']}")
    print(f"  Answer:    {ragas_dataset[0]['answer'][:200]}...")
    print(f"  Contexts:  {len(ragas_dataset[0]['contexts'])} chunks")

except ValueError as e:
    print(f"❌ {e}")
except Exception as e:
    print(f"❌ Failed to create dataset: {e}")

✅ RAGAS dataset created — 20 rows

Sample row:
  Question:  What is machine learning?
  Answer:    Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus...
  Contexts:  5 chunks


In [22]:
!pip install -U langchain-community

In [24]:
import importlib
import ragas.metrics
importlib.reload(ragas.metrics)

from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset

try:
    print("⏳ Setting up Gemini 2.0 flash for RAGAS...")

    ragas_llm = LangchainLLMWrapper(
        ChatGoogleGenerativeAI(model="gemini-2.0-flash")
    )
    ragas_embeddings = LangchainEmbeddingsWrapper(
        GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
    )

    # Create fresh metric instances
    from ragas.metrics import Faithfulness, AnswerRelevancy, ContextPrecision

    faith_metric = Faithfulness(llm=ragas_llm)
    relevancy_metric = AnswerRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    precision_metric = ContextPrecision(llm=ragas_llm)

    print("✅ Fresh metrics created with gemini-2.0-flash")

    ragas_dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truths": [[gt] for gt in ground_truths],
        "reference": ground_truths
    })

    print(f"✅ Dataset built — {len(ragas_dataset)} rows")
    print("⏳ Running RAGAS evaluation — this may take a few minutes...")

    results = evaluate(
        ragas_dataset,
        metrics=[faith_metric, relevancy_metric, precision_metric]
    )

    print("✅ Evaluation complete!")
    print(results)

except Exception as e:
    print(f"❌ RAGAS evaluation failed: {e}")

ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'

In [25]:
import sys
print(sys.version)

3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [26]:
!pip show ragas
!pip show langchain-community
!pip show langchain
!pip show langchain-core

Name: ragas
Version: 0.4.3
Summary: Evaluation framework for RAG and LLM applications
Home-page: https://github.com/vibrantlabsai/ragas
Author: 
Author-email: 
License: Apache License
                           Version 2.0, January 2004
                        http://www.apache.org/licenses/

   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION

   1. Definitions.

      "License" shall mean the terms and conditions for use, reproduction,
      and distribution as defined by Sections 1 through 9 of this document.

      "Licensor" shall mean the copyright owner or entity authorized by
      the copyright owner that is granting the License.

      "Legal Entity" shall mean the union of the acting entity and all
      other entities that control, are controlled by, or are under common
      control with that entity. For the purposes of this definition,
      "control" means (i) the power, direct or indirect, to cause the
      direction or management of such entity, whether by